In [68]:
import sys
from pathlib import Path
# Add the parent directory to sys.path so we can import synth_extract
sys.path.insert(0, str(Path.cwd().parent)) 

In [ ]:
import numpy as np
import pandas as pd

In [2]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

#### Working on the development set

In [ ]:
data_path = Path("../data/development_set/dataset_labels.csv")

In [ ]:
df = pd.read_csv(data_path)

In [ ]:
df.head()

In [ ]:
def dataset_stats(data, name):
    total = len(data)
    positives = (data["label"] == 1).sum()
    negatives = (data["label"] == 0).sum()

    uncertain = data["uncertain"].notna().sum()
    certain = data["uncertain"].isna().sum()

    certain_data = data[data["uncertain"].isna()]
    certain_positives = (certain_data["label"] == 1).sum()
    certain_negatives = (certain_data["label"] == 0).sum()

    return {
        "Dataset": name,
        "N": total,
        
        "Positive": positives,
        "Negative": negatives,
        "Uncertain": uncertain,
        "Certain": certain,
        "N after removing uncertain": len(certain_data),
        "Positive after removing uncertain": certain_positives,
        "Negative after removing uncertain": certain_negatives,
    }


stats = [
    dataset_stats(df, "Full"),
    dataset_stats(df[df["split"] == "train"], "Train"),
    dataset_stats(df[df["split"] == "test"], "Test"),
]

dataset_stats_table = pd.DataFrame(stats).set_index("Dataset")

dataset_stats_table

In [ ]:
qwen_label_columns = sorted(
    column for column in df.columns if column.startswith("qwen_s")
)
if not qwen_label_columns:
    raise ValueError("No model-label columns matching 'qwen_s*' were found")

uncertainty_column = next(
    (
        column
        for column in ("uncertainty", "uncertain")
        if column in df.columns
    ),
    None,
)
if uncertainty_column is None:
    raise ValueError(
        "No uncertainty column named 'uncertainty' or 'uncertain' was found"
    )

uncertainty_values = pd.to_numeric(
    df[uncertainty_column], errors="coerce"
)
scope_masks = {
    "all": pd.Series(True, index=df.index),
    # Missing/blank uncertainty is included because it is not equal to 1.
    "certain": uncertainty_values.ne(1),
}

metric_rows = []
for model_column in qwen_label_columns:
    for scope_name, scope_mask in scope_masks.items():
        for split_name in ("train", "test"):
            split_rows = df.loc[
                df["split"].eq(split_name) & scope_mask,
                ["label", model_column],
            ].apply(pd.to_numeric, errors="coerce").dropna()

            if split_rows.empty:
                raise ValueError(
                    f"No evaluable rows for {model_column!r} in "
                    f"{split_name!r}, scope={scope_name!r}"
                )

            y_true = split_rows["label"].astype(int)
            y_pred = split_rows[model_column].astype(int)
            observed_labels = set(y_true) | set(y_pred)
            if not observed_labels <= {0, 1}:
                raise ValueError(
                    f"Non-binary values for {model_column!r} in "
                    f"{split_name!r}: {sorted(observed_labels)}"
                )

            metric_rows.append(
                {
                    "qwen_label": model_column,
                    "scope": scope_name,
                    "split": split_name,
                    "samples": len(split_rows),
                    "accuracy": accuracy_score(y_true, y_pred),
                    "precision": precision_score(
                        y_true, y_pred, pos_label=1, zero_division=0
                    ),
                    "recall": recall_score(
                        y_true, y_pred, pos_label=1, zero_division=0
                    ),
                    "f1": f1_score(
                        y_true, y_pred, pos_label=1, zero_division=0
                    ),
                }
            )

qwen_metrics = pd.DataFrame(metric_rows)


In [ ]:
qwen_metrics.style.format(
    {
        "accuracy": "{:.3f}",
        "precision": "{:.3f}",
        "recall": "{:.3f}",
        "f1": "{:.3f}",
    }
).hide(axis="index")

In [ ]:
metric_order = ["accuracy", "precision", "recall", "f1"]
split_order = ["train", "test"]

def grouped_metric_table(scope_name):
    table = (
        qwen_metrics.loc[qwen_metrics["scope"].eq(scope_name)]
        .pivot(
            index="qwen_label",
            columns="split",
            values=metric_order,
        )
        .swaplevel(0, 1, axis=1)
        .reindex(
            columns=pd.MultiIndex.from_product(
                [split_order, metric_order],
                names=["dataset", "metric"],
            )
        )
    )
    table.index.name = "model"
    return table.rename(
        columns={
            "train": "Train",
            "test": "Test",
            "accuracy": "Accuracy",
            "precision": "Precision",
            "recall": "Recall",
            "f1": "F1",
        }
    )

qwen_metrics_table = grouped_metric_table("all")
qwen_metrics_table.style.format("{:.3f}").set_caption(
    "All datapoints"
).set_table_styles(
    [
        {
            "selector": "th.col_heading",
            "props": [("text-align", "center")],
        }
    ]
)

In [ ]:
qwen_certain_metrics_table = grouped_metric_table("certain")
certain_counts = {
    split_name: int(
        (df["split"].eq(split_name) & scope_masks["certain"]).sum()
    )
    for split_name in split_order
}
qwen_certain_metrics_table.style.format("{:.3f}").set_caption(
    f"Certain datapoints only ({uncertainty_column} != 1): "
    f"Train n={certain_counts['train']}, "
    f"Test n={certain_counts['test']}"
).set_table_styles(
    [
        {
            "selector": "th.col_heading",
            "props": [("text-align", "center")],
        }
    ]
)

In [ ]:
data_path = Path("../data/development_set/dataset_labels_category.csv")

In [ ]:
df = pd.read_csv(data_path)

In [ ]:
df.head()

In [ ]:
repetition_columns = {}
for column in df.columns:
    if "_K" not in column or "catcol" in column.lower():
        continue
    model_name, repetition_number = column.rsplit("_K", 1)
    if repetition_number in {"1", "2", "3"}:
        repetition_columns.setdefault(model_name, {})[
            f"K{repetition_number}"
        ] = column

if not repetition_columns:
    raise ValueError("No repeated model columns ending in _K1, _K2, or _K3 were found")

expected_repetitions = {"K1", "K2", "K3"}
for model_name, columns_by_repetition in repetition_columns.items():
    missing_repetitions = expected_repetitions - set(columns_by_repetition)
    if missing_repetitions:
        raise ValueError(
            f"{model_name!r} is missing repetitions: "
            f"{sorted(missing_repetitions)}"
        )

uncertainty_column = next(
    (column for column in ("uncertainty", "uncertain") if column in df.columns),
    None,
)
if uncertainty_column is None:
    raise ValueError(
        "No uncertainty column named 'uncertainty' or 'uncertain' was found"
    )

uncertainty_values = pd.to_numeric(df[uncertainty_column], errors="coerce")
category_scope_masks = {
    "all": pd.Series(True, index=df.index),
    "certain": uncertainty_values.ne(1),
}
category_metric_order = ["accuracy", "precision", "recall", "f1"]
category_split_order = ["train", "test"]

repetition_metric_rows = []
for model_name, columns_by_repetition in sorted(repetition_columns.items()):
    for repetition in ("K1", "K2", "K3"):
        prediction_column = columns_by_repetition[repetition]
        for scope_name, scope_mask in category_scope_masks.items():
            for split_name in category_split_order:
                split_rows = df.loc[
                    df["split"].eq(split_name) & scope_mask,
                    ["label", prediction_column],
                ].apply(pd.to_numeric, errors="coerce").dropna()

                if split_rows.empty:
                    raise ValueError(
                        f"No evaluable rows for {prediction_column!r} in "
                        f"{split_name!r}, scope={scope_name!r}"
                    )

                y_true = split_rows["label"].astype(int)
                y_pred = split_rows[prediction_column].astype(int)
                observed_labels = set(y_true) | set(y_pred)
                if not observed_labels <= {0, 1}:
                    raise ValueError(
                        f"Non-binary values for {prediction_column!r}: "
                        f"{sorted(observed_labels)}"
                    )

                repetition_metric_rows.append(
                    {
                        "model": model_name,
                        "repetition": repetition,
                        "scope": scope_name,
                        "split": split_name,
                        "samples": len(split_rows),
                        "accuracy": accuracy_score(y_true, y_pred),
                        "precision": precision_score(
                            y_true, y_pred, pos_label=1, zero_division=0
                        ),
                        "recall": recall_score(
                            y_true, y_pred, pos_label=1, zero_division=0
                        ),
                        "f1": f1_score(
                            y_true, y_pred, pos_label=1, zero_division=0
                        ),
                    }
                )

category_repetition_metrics = pd.DataFrame(repetition_metric_rows)

summary_rows = []
for (model_name, scope_name, split_name), group in category_repetition_metrics.groupby(
    ["model", "scope", "split"], sort=True
):
    summary_row = {
        "model": model_name,
        "scope": scope_name,
        "split": split_name,
        "samples": int(group["samples"].iloc[0]),
        "repetitions": len(group),
    }
    for metric in category_metric_order:
        summary_row[f"{metric}_mean"] = group[metric].mean()
        summary_row[f"{metric}_std"] = group[metric].std(ddof=1)
    summary_rows.append(summary_row)

category_metric_summary = pd.DataFrame(summary_rows)

In [ ]:
category_repetition_metrics.style.format(
    {metric: "{:.3f}" for metric in category_metric_order}
).hide(axis="index").set_caption(
    "Metrics for each repetition (K1, K2, K3)"
)

In [ ]:
def grouped_repetition_metric_table(scope_name):
    scope_summary = category_metric_summary.loc[
        category_metric_summary["scope"].eq(scope_name)
    ]
    model_order = sorted(scope_summary["model"].unique())
    table = pd.DataFrame(index=pd.Index(model_order, name="model"))

    for split_name in category_split_order:
        split_summary = scope_summary.loc[
            scope_summary["split"].eq(split_name)
        ].set_index("model")
        for metric in category_metric_order:
            means = split_summary[f"{metric}_mean"].reindex(model_order)
            standard_deviations = split_summary[f"{metric}_std"].reindex(
                model_order
            )
            table[(split_name.title(), metric.title())] = [
                f"{mean:.3f} ± {std:.3f}"
                for mean, std in zip(means, standard_deviations)
            ]

    table.columns = pd.MultiIndex.from_tuples(
        table.columns, names=["dataset", "metric"]
    )
    return table

category_metrics_table = grouped_repetition_metric_table("all")
category_metrics_table.style.set_caption(
    "All datapoints — mean ± sample SD across K1, K2, and K3"
).set_table_styles(
    [
        {
            "selector": "th.col_heading",
            "props": [("text-align", "center")],
        }
    ]
)

In [ ]:
category_certain_metrics_table = grouped_repetition_metric_table("certain")
category_certain_counts = {
    split_name: int(
        (df["split"].eq(split_name) & category_scope_masks["certain"]).sum()
    )
    for split_name in category_split_order
}
category_certain_metrics_table.style.set_caption(
    f"Certain datapoints only ({uncertainty_column} != 1) — "
    f"mean ± sample SD across K1, K2, and K3; "
    f"Train n={category_certain_counts['train']}, "
    f"Test n={category_certain_counts['test']}"
).set_table_styles(
    [
        {
            "selector": "th.col_heading",
            "props": [("text-align", "center")],
        }
    ]
)

#### Creating manifest unid files for classification handling

In [ ]:
# from contextlib import closing
# import math
# import sqlite3

# central_db_path = Path("../data/central_papers.db").resolve()
# uid_manifest_dir = Path("../data/uid_manifest").resolve()
# uids_per_file = 10_000
# manifest_prefix = "classify_uids_"

# if not central_db_path.is_file():
#     raise FileNotFoundError(f"Central database not found: {central_db_path}")
# uid_manifest_dir.mkdir(parents=True, exist_ok=True)

# # Remove temporary files left by an interrupted earlier generation.
# for temporary_path in uid_manifest_dir.glob(f"{manifest_prefix}*.txt.tmp"):
#     temporary_path.unlink()

# database_uri = f"{central_db_path.as_uri()}?mode=ro"
# manifest_rows = []
# staged_files = []
# written_uid_count = 0

# with closing(sqlite3.connect(database_uri, uri=True, timeout=60)) as conn:
#     conn.execute("PRAGMA query_only = ON")
#     conn.execute("PRAGMA busy_timeout = 60000")

#     table_exists = conn.execute(
#         """
#         SELECT 1
#         FROM sqlite_master
#         WHERE type = 'table' AND name = 'classify'
#         """
#     ).fetchone()
#     if table_exists is None:
#         raise ValueError("central_papers.db has no classify table")

#     available_columns = {
#         row[1] for row in conn.execute("PRAGMA table_info(classify)")
#     }
#     required_columns = {"paper_id", "paper_uid"}
#     missing_columns = sorted(required_columns - available_columns)
#     if missing_columns:
#         raise ValueError(
#             "classify is missing column(s): " + ", ".join(missing_columns)
#         )

#     total_uid_count, unique_uid_count = conn.execute(
#         "SELECT COUNT(*), COUNT(DISTINCT paper_uid) FROM classify"
#     ).fetchone()
#     if total_uid_count != unique_uid_count:
#         raise ValueError(
#             f"classify contains {total_uid_count - unique_uid_count:,} "
#             "duplicate paper_uid value(s)"
#         )

#     cursor = conn.execute(
#         "SELECT paper_uid FROM classify ORDER BY paper_id"
#     )
#     file_number = 0
#     while True:
#         rows = cursor.fetchmany(uids_per_file)
#         if not rows:
#             break

#         uids = [str(row[0]).strip() for row in rows]
#         if any(not uid for uid in uids):
#             raise ValueError("classify contains an empty paper_uid")

#         file_number += 1
#         final_path = uid_manifest_dir / (
#             f"{manifest_prefix}{file_number:04d}.txt"
#         )
#         temporary_path = final_path.with_suffix(".txt.tmp")
#         temporary_path.write_text(
#             "\n".join(uids) + "\n",
#             encoding="utf-8",
#         )
#         staged_files.append((temporary_path, final_path))
#         written_uid_count += len(uids)
#         manifest_rows.append(
#             {
#                 "file": final_path.name,
#                 "uids": len(uids),
#                 "first_uid": uids[0],
#                 "last_uid": uids[-1],
#             }
#         )

# expected_file_count = math.ceil(total_uid_count / uids_per_file)
# if written_uid_count != total_uid_count:
#     raise RuntimeError(
#         f"Expected {total_uid_count:,} UIDs, wrote {written_uid_count:,}"
#     )
# if len(staged_files) != expected_file_count:
#     raise RuntimeError(
#         f"Expected {expected_file_count:,} files, staged {len(staged_files):,}"
#     )
# if any(row["uids"] > uids_per_file for row in manifest_rows):
#     raise RuntimeError("A manifest exceeds the 10,000-UID limit")

# # Replace only manifests created by this cell; leave unrelated files alone.
# for old_path in uid_manifest_dir.glob(f"{manifest_prefix}*.txt"):
#     old_path.unlink()
# for temporary_path, final_path in staged_files:
#     temporary_path.replace(final_path)

# uid_manifest_summary = pd.DataFrame(manifest_rows)
# print(f"Manifest directory: {uid_manifest_dir}")
# print(f"UIDs written: {written_uid_count:,}")
# print(f"Manifest files: {len(staged_files):,}")
# print(f"Maximum UIDs per file: {uids_per_file:,}")
# uid_manifest_summary

Manifest directory: /Users/kevinge/Work/Data Extraction/synth_extract/data/uid_manifest
UIDs written: 1,012,267
Manifest files: 102
Maximum UIDs per file: 10,000


,file,uids,first_uid,last_uid
0,classify_uids_0001.txt,10000,ID000000002,ID000011897
1,classify_uids_0002.txt,10000,ID000011898,ID000029643
2,classify_uids_0003.txt,10000,ID000029644,ID000039661
3,classify_uids_0004.txt,10000,ID000039662,ID000049831
4,classify_uids_0005.txt,10000,ID000049832,ID000059926
...,...,...,...,...
97,classify_uids_0098.txt,10000,ID001055120,ID001066875
98,classify_uids_0099.txt,10000,ID001066876,ID001078146
99,classify_uids_0100.txt,10000,ID001078147,ID001088423
100,classify_uids_0101.txt,10000,ID001088424,ID001099501


#### Checking Status of Classification

In [61]:
from contextlib import closing
import sqlite3

central_db_path = Path("../data/central_papers.db").resolve()
uid_manifest_dir = Path("../data/uid_manifest").resolve()
manifest_paths = sorted(uid_manifest_dir.glob("classify_uids_*.txt"))

if not central_db_path.is_file():
    raise FileNotFoundError(f"Central database not found: {central_db_path}")
if not manifest_paths:
    raise FileNotFoundError(
        f"No classify UID manifests found in: {uid_manifest_dir}"
    )

def quote_sqlite_identifier(identifier):
    return '"' + identifier.replace('"', '""') + '"'

def result_column_sort_key(column):
    suffix = column.removeprefix("class_run_")
    return (0, int(suffix)) if suffix.isdigit() else (1, column)

database_uri = f"{central_db_path.as_uri()}?mode=ro"
manifest_status_rows = []

with closing(sqlite3.connect(database_uri, uri=True, timeout=60)) as conn:
    conn.execute("PRAGMA busy_timeout = 60000")
    conn.execute("PRAGMA temp_store = MEMORY")

    table_exists = conn.execute(
        """
        SELECT 1
        FROM sqlite_master
        WHERE type = 'table' AND name = 'classify'
        """
    ).fetchone()
    if table_exists is None:
        raise ValueError("central_papers.db has no classify table")

    classify_columns = [
        row[1] for row in conn.execute("PRAGMA table_info(classify)")
    ]
    result_columns = sorted(
        (column for column in classify_columns if column.startswith("class_run_")),
        key=result_column_sort_key,
    )
    if not result_columns:
        raise ValueError("classify has no columns matching 'class_run_*'")

    total_classify_rows = conn.execute(
        "SELECT COUNT(*) FROM classify"
    ).fetchone()[0]
    overall_status_rows = []
    for result_column in result_columns:
        column_sql = quote_sqlite_identifier(result_column)
        completed, missing = conn.execute(
            f"""
            SELECT
                SUM({column_sql} IS NOT NULL),
                SUM({column_sql} IS NULL)
            FROM classify
            """
        ).fetchone()
        overall_status_rows.append(
            {
                "result_column": result_column,
                "total": total_classify_rows,
                "completed": int(completed or 0),
                "missing": int(missing or 0),
                "completed_pct": (
                    100 * int(completed or 0) / total_classify_rows
                    if total_classify_rows
                    else 0.0
                ),
            }
        )

    conn.execute(
        """
        CREATE TEMP TABLE manifest_uids (
            paper_uid TEXT PRIMARY KEY
        ) WITHOUT ROWID
        """
    )

    for manifest_path in manifest_paths:
        uids = [
            line.strip()
            for line in manifest_path.read_text(
                encoding="utf-8-sig"
            ).splitlines()
            if line.strip() and not line.lstrip().startswith("#")
        ]
        if not uids:
            raise ValueError(f"Manifest is empty: {manifest_path}")
        if len(uids) != len(set(uids)):
            raise ValueError(
                f"Manifest contains duplicate UIDs: {manifest_path}"
            )

        conn.execute("DELETE FROM manifest_uids")
        conn.executemany(
            "INSERT INTO manifest_uids(paper_uid) VALUES (?)",
            ((uid,) for uid in uids),
        )

        status_row = {
            "manifest": manifest_path.name,
            "uids": len(uids),
        }

        for result_column in result_columns:
            column_sql = quote_sqlite_identifier(result_column)
            completed, missing = conn.execute(
                f"""
                SELECT
                    SUM(classify.{column_sql} IS NOT NULL),
                    SUM(classify.{column_sql} IS NULL)
                FROM manifest_uids AS manifest
                JOIN classify
                    ON classify.paper_uid = manifest.paper_uid
                """
            ).fetchone()
            completed = int(completed or 0)
            missing = int(missing or 0)
            status_row[f"{result_column}_completed"] = completed
            status_row[f"{result_column}_missing"] = missing
            status_row[f"{result_column}_completed_pct"] = (
                100 * completed / len(uids)
            )

        manifest_status_rows.append(status_row)

overall_classification_status = pd.DataFrame(overall_status_rows)
manifest_classification_status = pd.DataFrame(manifest_status_rows)

print(f"Database: {central_db_path}")
print(f"classify rows: {total_classify_rows:,}")
print(f"Manifests checked: {len(manifest_paths):,}")
print(f"Manifest UIDs checked: {manifest_classification_status['uids'].sum():,}")
display(
    overall_classification_status.style.format(
        {
            "total": "{:,}",
            "completed": "{:,}",
            "missing": "{:,}",
            "completed_pct": "{:.2f}%",
        }
    ).hide(axis="index").set_caption("Overall classification status")
)

percentage_formats = {
    f"{column}_completed_pct": "{:.2f}%"
    for column in result_columns
}
count_formats = {
    f"{column}_{status}": "{:,}"
    for column in result_columns
    for status in ("completed", "missing")
}
display(
    manifest_classification_status.style.format(
        {
            "uids": "{:,}",
            **count_formats,
            **percentage_formats,
        }
    ).hide(axis="index").set_caption(
        "Classification completion by UID manifest"
    )
)

Database: /Users/kevinge/Work/Data Extraction/synth_extract/data/central_papers.db
classify rows: 1,012,267
Manifests checked: 5
Manifest UIDs checked: 23,829


result_column,total,completed,missing,completed_pct
class_run_1,"1,012,267","1,012,267",0,100.00%
class_run_2,"1,012,267","1,012,267",0,100.00%
class_run_3,"1,012,267","1,012,267",0,100.00%


manifest,uids,class_run_1_completed,class_run_1_missing,class_run_1_completed_pct,class_run_2_completed,class_run_2_missing,class_run_2_completed_pct,class_run_3_completed,class_run_3_missing,class_run_3_completed_pct
classify_uids_0001.txt,"5,000","5,000",0,100.00%,"5,000",0,100.00%,"5,000",0,100.00%
classify_uids_0002.txt,"5,000","5,000",0,100.00%,"5,000",0,100.00%,"5,000",0,100.00%
classify_uids_0003.txt,"5,000","5,000",0,100.00%,"5,000",0,100.00%,"5,000",0,100.00%
classify_uids_0004.txt,"5,000","5,000",0,100.00%,"5,000",0,100.00%,"5,000",0,100.00%
classify_uids_0005.txt,"3,829","3,829",0,100.00%,"3,829",0,100.00%,"3,829",0,100.00%


In [62]:
from contextlib import closing
import sqlite3

central_db_path = Path("../data/central_papers.db").resolve()
if not central_db_path.is_file():
    raise FileNotFoundError(f"Central database not found: {central_db_path}")

def quote_sqlite_identifier(identifier):
    return '"' + identifier.replace('"', '""') + '"'

database_uri = f"{central_db_path.as_uri()}?mode=ro"
prediction_count_rows = []

with closing(sqlite3.connect(database_uri, uri=True, timeout=60)) as conn:
    conn.execute("PRAGMA query_only = ON")
    conn.execute("PRAGMA busy_timeout = 60000")

    classify_columns = [
        row[1] for row in conn.execute("PRAGMA table_info(classify)")
    ]
    result_columns = sorted(
        (column for column in classify_columns if column.startswith("class_run_")),
        key=lambda column: (
            int(column.removeprefix("class_run_"))
            if column.removeprefix("class_run_").isdigit()
            else float("inf")
        ),
    )
    if not result_columns:
        raise ValueError("classify has no columns matching 'class_run_*'")

    for result_column in result_columns:
        column_sql = quote_sqlite_identifier(result_column)
        total, positive, negative, null = conn.execute(
            f"""
            SELECT
                COUNT(*),
                SUM({column_sql} = 1),
                SUM({column_sql} = 0),
                SUM({column_sql} IS NULL)
            FROM classify
            """
        ).fetchone()
        positive = int(positive or 0)
        negative = int(negative or 0)
        null = int(null or 0)
        classified = positive + negative
        prediction_count_rows.append(
            {
                "result_column": result_column,
                "total": total,
                "classified": classified,
                "positive": positive,
                "negative": negative,
                "null": null,
                "positive_pct_of_classified": (
                    100 * positive / classified if classified else 0.0
                ),
            }
        )

prediction_counts = pd.DataFrame(prediction_count_rows)
print(f"Database: {central_db_path}")
display(
    prediction_counts.style.format(
        {
            "total": "{:,}",
            "classified": "{:,}",
            "positive": "{:,}",
            "negative": "{:,}",
            "null": "{:,}",
            "positive_pct_of_classified": "{:.2f}%",
        }
    ).hide(axis="index").set_caption(
        "Positive, negative, and NULL predictions in classify"
    )
)

Database: /Users/kevinge/Work/Data Extraction/synth_extract/data/central_papers.db


result_column,total,classified,positive,negative,null,positive_pct_of_classified
class_run_1,"1,012,267","1,012,199","273,004","739,195",0,26.97%
class_run_2,"1,012,267","1,012,199","272,995","739,204",0,26.97%
class_run_3,"1,012,267","1,012,199","273,191","739,008",0,26.99%


In [63]:
from contextlib import closing
from pathlib import Path
import sqlite3

central_db_path = Path("../data/central_papers.db").resolve()
if not central_db_path.is_file():
    raise FileNotFoundError(f"Central database not found: {central_db_path}")

database_uri = f"{central_db_path.as_uri()}?mode=ro"
with closing(sqlite3.connect(database_uri, uri=True, timeout=60)) as conn:
    conn.execute("PRAGMA query_only = ON")
    conn.execute("PRAGMA busy_timeout = 60000")

    classify_columns = {
        row[1] for row in conn.execute("PRAGMA table_info(classify)")
    }
    required_columns = {"class_run_1", "class_run_2", "class_run_3"}
    missing_columns = sorted(required_columns - classify_columns)
    if missing_columns:
        raise ValueError(
            "classify is missing column(s): " + ", ".join(missing_columns)
        )

    (total_rows, fully_evaluated, incomplete, unanimous_positive,
     unanimous_negative, unanimous_failed, different) = conn.execute(
        """
        SELECT
            COUNT(*),
            SUM(
                class_run_1 IS NOT NULL
                AND class_run_2 IS NOT NULL
                AND class_run_3 IS NOT NULL
            ),
            SUM(
                class_run_1 IS NULL
                OR class_run_2 IS NULL
                OR class_run_3 IS NULL
            ),
            SUM(
                class_run_1 = 1
                AND class_run_2 = 1
                AND class_run_3 = 1
            ),
            SUM(
                class_run_1 = 0
                AND class_run_2 = 0
                AND class_run_3 = 0
            ),
            SUM(
                class_run_1 = 'FAILED'
                AND class_run_2 = 'FAILED'
                AND class_run_3 = 'FAILED'
            ),
            SUM(
                class_run_1 IS NOT NULL
                AND class_run_2 IS NOT NULL
                AND class_run_3 IS NOT NULL
                AND NOT (
                    class_run_1 IS class_run_2
                    AND class_run_2 IS class_run_3
                )
            )
        FROM classify
        """
    ).fetchone()

    run_1_failed, run_2_failed, run_3_failed, any_failed = conn.execute(
        """
        SELECT
            SUM(class_run_1 = 'FAILED'),
            SUM(class_run_2 = 'FAILED'),
            SUM(class_run_3 = 'FAILED'),
            SUM(
                class_run_1 = 'FAILED'
                OR class_run_2 = 'FAILED'
                OR class_run_3 = 'FAILED'
            )
        FROM classify
        """
    ).fetchone()

    pattern_rows = conn.execute(
        """
        SELECT class_run_1, class_run_2, class_run_3, COUNT(*) AS rows
        FROM classify
        GROUP BY class_run_1, class_run_2, class_run_3
        ORDER BY rows DESC
        """
    ).fetchall()

total_rows = int(total_rows or 0)
fully_evaluated = int(fully_evaluated or 0)
incomplete = int(incomplete or 0)
unanimous_positive = int(unanimous_positive or 0)
unanimous_negative = int(unanimous_negative or 0)
unanimous_failed = int(unanimous_failed or 0)
different = int(different or 0)
all_same = unanimous_positive + unanimous_negative + unanimous_failed
run_1_failed = int(run_1_failed or 0)
run_2_failed = int(run_2_failed or 0)
run_3_failed = int(run_3_failed or 0)
any_failed = int(any_failed or 0)
partially_failed = any_failed - unanimous_failed

agreement_rows = [
    {"status": "All three = 1", "rows": unanimous_positive},
    {"status": "All three = 0", "rows": unanimous_negative},
    {"status": "All three = FAILED", "rows": unanimous_failed},
    {"status": "All three agree", "rows": all_same},
    {"status": "Different (all non-NULL)", "rows": different},
    {"status": "Incomplete (at least one NULL)", "rows": incomplete},
]
for row in agreement_rows:
    row["pct_of_all_rows"] = (
        100 * row["rows"] / total_rows if total_rows else 0.0
    )

agreement_statistics = pd.DataFrame(agreement_rows)
failed_consistency = pd.DataFrame(
    [
        {"check": "FAILED in run 1", "rows": run_1_failed},
        {"check": "FAILED in run 2", "rows": run_2_failed},
        {"check": "FAILED in run 3", "rows": run_3_failed},
        {"check": "FAILED in at least one run", "rows": any_failed},
        {"check": "FAILED in all three runs", "rows": unanimous_failed},
        {"check": "FAILED in only some runs", "rows": partially_failed},
    ]
)

def display_run_value(value):
    return "NULL" if value is None else str(value)

agreement_patterns = pd.DataFrame(
    [
        {
            "class_run_1": display_run_value(run_1),
            "class_run_2": display_run_value(run_2),
            "class_run_3": display_run_value(run_3),
            "status": (
                "incomplete"
                if None in (run_1, run_2, run_3)
                else "same"
                if run_1 == run_2 == run_3
                else "different"
            ),
            "rows": rows,
            "pct_of_all_rows": 100 * rows / total_rows if total_rows else 0.0,
        }
        for run_1, run_2, run_3, rows in pattern_rows
    ]
)

print(f"Database: {central_db_path}")
print(f"Total rows: {total_rows:,}")
print(f"Fully evaluated rows: {fully_evaluated:,}")
print(
    f"Agreement among fully evaluated rows: "
    f"{(100 * all_same / fully_evaluated if fully_evaluated else 0.0):.2f}%"
)
display(
    agreement_statistics.style.format(
        {"rows": "{:,}", "pct_of_all_rows": "{:.2f}%"}
    ).hide(axis="index").set_caption("Agreement across all three runs")
)
display(
    failed_consistency.style.format(
        {"rows": "{:,}"}
    ).hide(axis="index").set_caption("FAILED consistency check")
)
display(
    agreement_patterns.style.format(
        {"rows": "{:,}", "pct_of_all_rows": "{:.2f}%"}
    ).hide(axis="index").set_caption("Detailed run-value combinations")
)

if partially_failed == 0:
    print("PASSED: every row containing FAILED is FAILED in all three runs.")
else:
    print(
        f"FAILED CHECK VIOLATION: {partially_failed:,} rows contain FAILED "
        "in only some runs."
    )

Database: /Users/kevinge/Work/Data Extraction/synth_extract/data/central_papers.db
Total rows: 1,012,267
Fully evaluated rows: 1,012,267
Agreement among fully evaluated rows: 97.65%


status,rows,pct_of_all_rows
All three = 1,"261,085",25.79%
All three = 0,"727,285",71.85%
All three = FAILED,68,0.01%
All three agree,"988,438",97.65%
Different (all non-NULL),"23,829",2.35%
Incomplete (at least one NULL),0,0.00%


check,rows
FAILED in run 1,68
FAILED in run 2,68
FAILED in run 3,68
FAILED in at least one run,68
FAILED in all three runs,68
FAILED in only some runs,0


class_run_1,class_run_2,class_run_3,status,rows,pct_of_all_rows
0,0,0,same,"727,285",71.85%
1,1,1,same,"261,085",25.79%
0,1,1,different,"6,069",0.60%
1,0,1,different,"6,037",0.60%
1,0,0,different,"5,882",0.58%
0,1,0,different,"5,841",0.58%
FAILED,FAILED,FAILED,same,68,0.01%


PASSED: every row containing FAILED is FAILED in all three runs.


In [65]:
from contextlib import closing
from pathlib import Path
import sqlite3

central_db_path = Path("../data/central_papers.db").resolve()
if not central_db_path.is_file():
    raise FileNotFoundError(f"Central database not found: {central_db_path}")

database_uri = f"{central_db_path.as_uri()}?mode=ro"
with closing(sqlite3.connect(database_uri, uri=True, timeout=60)) as conn:
    conn.execute("PRAGMA query_only = ON")
    conn.execute("PRAGMA busy_timeout = 60000")

    classify_columns = {
        row[1] for row in conn.execute("PRAGMA table_info(classify)")
    }
    if "class_label" not in classify_columns:
        raise ValueError(
            "classify has no class_label column. Run the majority-vote cell first."
        )

    positive, negative, failed, null, total = conn.execute(
        """
        SELECT
            SUM(class_label = 1),
            SUM(class_label = 0),
            SUM(class_label = 'FAILED'),
            SUM(class_label IS NULL),
            COUNT(*)
        FROM classify
        """
    ).fetchone()

positive = int(positive or 0)
negative = int(negative or 0)
failed = int(failed or 0)
null = int(null or 0)
total = int(total or 0)

class_label_statistics = pd.DataFrame(
    [
        {"class_label": "Positive (1)", "rows": positive},
        {"class_label": "Negative (0)", "rows": negative},
        {"class_label": "FAILED", "rows": failed},
        {"class_label": "NULL", "rows": null},
    ]
)
class_label_statistics["pct_of_total"] = (
    100 * class_label_statistics["rows"] / total if total else 0.0
)

if positive + negative + failed + null != total:
    raise RuntimeError("class_label counts do not add up to the table total")

print(f"Database: {central_db_path}")
print(f"Total rows: {total:,}")
display(
    class_label_statistics.style.format(
        {"rows": "{:,}", "pct_of_total": "{:.2f}%"}
    ).hide(axis="index").set_caption("Final class_label distribution")
)

Database: /Users/kevinge/Work/Data Extraction/synth_extract/data/central_papers.db
Total rows: 1,012,267


class_label,rows,pct_of_total
Positive (1),"273,191",26.99%
Negative (0),"739,008",73.01%
FAILED,68,0.01%
NULL,0,0.00%


In [71]:
import synth_extract.utils.sql_helpers as sql

In [72]:
sql.list_tables("../data/central_papers.db")

['classify', 'downloaded_papers', 'papers']

In [73]:
sql.list_tables("../data/central_workspace.db")

['papers']

In [75]:
sql.get_table_schema("../data/central_papers.db", "papers")

,cid,name,type,notnull,dflt_value,pk
0,0,paper_id,INTEGER,0,None,1
1,1,paper_uid,TEXT,1,None,0
2,2,identifier_type,TEXT,1,None,0
3,3,identifier_value,TEXT,1,None,0
4,4,doi,TEXT,0,None,0
5,5,arxiv_id,TEXT,0,None,0
6,6,pmcid,TEXT,0,None,0
7,7,title,TEXT,0,None,0
8,8,abstract,TEXT,0,None,0
9,9,sources,TEXT,1,None,0


In [74]:
sql.get_table_schema("../data/central_papers.db", "classify")

,cid,name,type,notnull,dflt_value,pk
0,0,paper_id,INTEGER,0,None,1
1,1,paper_uid,TEXT,1,None,0
2,2,canonical_source,TEXT,1,None,0
3,3,fulltext_path,TEXT,1,None,0
4,4,class_run_1,INTEGER,0,NULL,0
5,5,class_run_2,INTEGER,0,NULL,0
6,6,class_run_3,INTEGER,0,NULL,0
7,7,class_label,INTEGER,0,NULL,0


In [78]:
sql.get_table_schema("../data/central_papers.db", "polymer_material_papers")

,cid,name,type,notnull,dflt_value,pk
0,0,paper_id,INTEGER,0,None,1
1,1,paper_uid,TEXT,1,None,0
2,2,identifier_type,TEXT,1,None,0
3,3,identifier_value,TEXT,1,None,0
4,4,doi,TEXT,0,None,0
5,5,arxiv_id,TEXT,0,None,0
6,6,pmcid,TEXT,0,None,0
7,7,title,TEXT,0,None,0
8,8,class_label,INTEGER,0,NULL,0
9,9,sources,TEXT,1,None,0


In [80]:
from contextlib import closing
from pathlib import Path
import math
import sqlite3

import pandas as pd

central_db_path = Path("../data/central_papers.db").resolve()
uid_manifest_dir = Path("../data/uid_manifest").resolve()
uids_per_file = 3_000
manifest_prefix = "category_uids_"

if not central_db_path.is_file():
    raise FileNotFoundError(f"Central database not found: {central_db_path}")
uid_manifest_dir.mkdir(parents=True, exist_ok=True)

# Remove temporary files left by an interrupted earlier generation.
for temporary_path in uid_manifest_dir.glob(f"{manifest_prefix}*.txt.tmp"):
    temporary_path.unlink()

database_uri = f"{central_db_path.as_uri()}?mode=ro"
manifest_rows = []
staged_files = []
written_uid_count = 0

with closing(sqlite3.connect(database_uri, uri=True, timeout=60)) as conn:
    conn.execute("PRAGMA query_only = ON")
    conn.execute("PRAGMA busy_timeout = 60000")

    table_exists = conn.execute(
        """
        SELECT 1
        FROM sqlite_master
        WHERE type = 'table' AND name = 'polymer_material_papers'
        """
    ).fetchone()
    if table_exists is None:
        raise ValueError(
            "central_papers.db has no polymer_material_papers table"
        )

    available_columns = {
        row[1]
        for row in conn.execute(
            "PRAGMA table_info(polymer_material_papers)"
        )
    }
    required_columns = {"paper_id", "paper_uid", "category_class"}
    missing_columns = sorted(required_columns - available_columns)
    if missing_columns:
        raise ValueError(
            "polymer_material_papers is missing column(s): "
            + ", ".join(missing_columns)
        )

    pending_uid_count, unique_uid_count = conn.execute(
        """
        SELECT COUNT(*), COUNT(DISTINCT paper_uid)
        FROM polymer_material_papers
        WHERE category_class IS NULL
        """
    ).fetchone()
    if pending_uid_count != unique_uid_count:
        raise ValueError(
            f"Pending rows contain "
            f"{pending_uid_count - unique_uid_count:,} duplicate UID(s)"
        )

    cursor = conn.execute(
        """
        SELECT paper_uid
        FROM polymer_material_papers
        WHERE category_class IS NULL
        ORDER BY paper_id
        """
    )
    file_number = 0
    while True:
        rows = cursor.fetchmany(uids_per_file)
        if not rows:
            break

        uids = [str(row[0]).strip() for row in rows]
        if any(not uid for uid in uids):
            raise ValueError(
                "polymer_material_papers contains an empty paper_uid"
            )

        file_number += 1
        final_path = uid_manifest_dir / (
            f"{manifest_prefix}{file_number:04d}.txt"
        )
        temporary_path = final_path.with_suffix(".txt.tmp")
        temporary_path.write_text(
            "\n".join(uids) + "\n",
            encoding="utf-8",
        )
        staged_files.append((temporary_path, final_path))
        written_uid_count += len(uids)
        manifest_rows.append(
            {
                "file": final_path.name,
                "uids": len(uids),
                "first_uid": uids[0],
                "last_uid": uids[-1],
            }
        )

expected_file_count = math.ceil(pending_uid_count / uids_per_file)
if written_uid_count != pending_uid_count:
    raise RuntimeError(
        f"Expected {pending_uid_count:,} UIDs, wrote {written_uid_count:,}"
    )
if len(staged_files) != expected_file_count:
    raise RuntimeError(
        f"Expected {expected_file_count:,} files, staged "
        f"{len(staged_files):,}"
    )
if any(row["uids"] > uids_per_file for row in manifest_rows):
    raise RuntimeError("A category manifest exceeds the 3,000-UID limit")

# Replace only category manifests; unrelated UID files are left untouched.
for old_path in uid_manifest_dir.glob(f"{manifest_prefix}*.txt"):
    old_path.unlink()
for temporary_path, final_path in staged_files:
    temporary_path.replace(final_path)

category_manifest_summary = pd.DataFrame(
    manifest_rows,
    columns=["file", "uids", "first_uid", "last_uid"],
)
print(f"Database: {central_db_path}")
print("Table: polymer_material_papers")
print("Filter: category_class IS NULL")
print(f"Manifest directory: {uid_manifest_dir}")
print(f"UIDs written: {written_uid_count:,}")
print(f"Manifest files: {len(staged_files):,}")
print(f"Maximum UIDs per file: {uids_per_file:,}")
category_manifest_summary

Database: /Users/kevinge/Work/Data Extraction/synth_extract/data/central_papers.db
Table: polymer_material_papers
Filter: category_class IS NULL
Manifest directory: /Users/kevinge/Work/Data Extraction/synth_extract/data/uid_manifest
UIDs written: 273,191
Manifest files: 92
Maximum UIDs per file: 3,000


,file,uids,first_uid,last_uid
0,category_uids_0001.txt,3000,ID000000608,ID000005905
1,category_uids_0002.txt,3000,ID000005906,ID000011088
2,category_uids_0003.txt,3000,ID000011089,ID000027040
3,category_uids_0004.txt,3000,ID000027042,ID000033604
4,category_uids_0005.txt,3000,ID000033608,ID000041529
...,...,...,...,...
87,category_uids_0088.txt,3000,ID001031771,ID001037960
88,category_uids_0089.txt,3000,ID001037965,ID001044028
89,category_uids_0090.txt,3000,ID001044029,ID001072084
90,category_uids_0091.txt,3000,ID001072085,ID001099985
